# 🎮 Analyse Stratégique Globale du Marché Steam - Projet Ubisoft

## 📝 Contexte et Objectifs
Dans le cadre du développement de notre nouveau titre AAA, la direction d'Ubisoft souhaite une analyse approfondie de l'écosystème Steam actuel. Ce rapport d'analyse exploratoire (EDA) vise à identifier les facteurs clés de succès et les tendances du marché.

Nous répondrons aux axes d'analyse suivants :
1.  **Macro-Analyse :** Dynamique temporelle, distribution des prix, langues et restrictions d'âge.
2.  **Performance & Qualité :** Analyse des notes joueurs (Ratings) et des éditeurs leaders.
3.  **Analyse des Genres :** Rentabilité, appréciation critique et spécialisation des éditeurs.
4.  **Stratégie Technique :** Couverture des plateformes (OS) par genre.

## ⚙️ Stack Technique
* **Source :** Data Lake S3 (JSON semi-structuré).
* **Moteur :** PySpark (Cluster Databricks).

In [0]:
# -----------------------------------------------------------------------------
# PHASE 1 : INGESTION ET PRÉPARATION DES DONNÉES (ETL)
# -----------------------------------------------------------------------------

# Ajout de 'regexp_extract' pour nettoyer les champs textes contenant des chiffres (Age)
from pyspark.sql.functions import (col, explode, split, try_to_date, year, count, 
                                   desc, avg, sum as _sum, when, lit, round, regexp_extract)

# 1. Chargement depuis le Data Lake S3
file_path = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
print("🔄 Ingestion des données brutes...")
df_raw = spark.read.json(file_path)

# 2. Aplatissement et Sélection (Data Cleaning)
df_flat = df_raw.select(
    col("data.name").alias("title"),
    col("data.developer").alias("developer"),
    col("data.publisher").alias("publisher"),
    col("data.genre").alias("genre_string"),
    col("data.initialprice").alias("price_raw"),
    col("data.release_date").alias("date_raw"),
    col("data.required_age").alias("required_age_raw"), # On garde le brut temporairement
    col("data.languages").alias("languages_string"),
    col("data.positive").alias("positive_reviews"),
    col("data.negative").alias("negative_reviews"),
    col("data.platforms").alias("platforms")
)

# 3. Transformations et Feature Engineering
df_clean = df_flat.withColumn(
    "release_year", year(try_to_date(col("date_raw"), "yyyy/MM/dd"))
).withColumn(
    "price_eur", col("price_raw").cast("double") / 100
).withColumn(
    # NETTOYAGE AGE : On extrait uniquement les chiffres présents dans la chaine
    # Ex: "MA 15+" devient "15", "18" devient "18"
    "required_age", regexp_extract(col("required_age_raw"), r"(\d+)", 1).cast("integer")
).withColumn(
    # Gestion des Nulls générés par l'extraction (si pas d'âge, on met 0)
    "required_age", when(col("required_age").isNull(), 0).otherwise(col("required_age"))
).withColumn(
    "total_reviews", col("positive_reviews") + col("negative_reviews")
).withColumn(
    "rating_pct", when(col("total_reviews") > 0, 
                       round((col("positive_reviews") / col("total_reviews")) * 100, 2)
                  ).otherwise(0)
)

# 4. Création de vues spécialisées

# A. Vue GENRES
df_genres_exploded = df_clean.select(
    "title", "price_eur", "rating_pct", "total_reviews", "platforms",
    explode(split(col("genre_string"), ", ")).alias("genre")
)

# B. Vue LANGUES
df_languages = df_clean.select(
    "title",
    explode(split(col("languages_string"), ", ")).alias("language")
)

print(f"✅ Pipeline ETL terminé. {df_clean.count()} jeux prêts. Colonne Âge nettoyée.")
display(df_clean.select("title", "required_age", "required_age_raw").limit(5))

🔄 Ingestion des données brutes...
✅ Pipeline ETL terminé. 55691 jeux prêts. Colonne Âge nettoyée.


title,required_age,required_age_raw
Counter-Strike,0,0
ASCENXION,0,0
Crown Trick,0,0
"Cook, Serve, Delicious! 3?!",0,0
细胞战争,0,0


## 🌍 Macro-Analyse : Production et Acteurs

**Questions :**
* Y a-t-il eu un effet "Covid" sur les sorties ?
* Quels sont les éditeurs/développeurs les plus actifs ?

In [0]:
# --- 1. Dynamique Temporelle ---
# Analyse du volume de sorties par an pour identifier les pics (2020-2022)
df_years = df_clean.filter(col("release_year").isNotNull()) \
                   .groupBy("release_year") \
                   .count() \
                   .orderBy(col("release_year").desc())

print("📊 Sorties par année :")
display(df_years)
# 👉 Action : Line Chart. X='release_year', Y='count'

# --- 2. Top Éditeurs (Publishers) ---
# L'énoncé demande spécifiquement les publishers (pas les developers)
df_top_publishers = df_clean.filter(col("publisher").isNotNull()) \
                            .groupBy("publisher") \
                            .count() \
                            .orderBy(desc("count")) \
                            .limit(15)

print("🏆 Top 15 Éditeurs (Publishers) :")
display(df_top_publishers)
# 👉 Action : Bar Chart. X='publisher', Y='count'

# --- 3. Top Développeurs ---
df_top_devs = df_clean.filter(col("developer").isNotNull()) \
                      .groupBy("developer") \
                      .count() \
                      .orderBy(desc("count")) \
                      .limit(10)

print("🔧 Top 10 Développeurs :")
display(df_top_devs)

📊 Sorties par année :


release_year,count
2022,5301
2021,6266
2020,6059
2019,5053
2018,5416
2017,4352
2016,2928
2015,1815
2014,1087
2013,330


Databricks visualization. Run in Databricks to view.

🏆 Top 15 Éditeurs (Publishers) :


publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
HH-Games,132
,132
Sekai Project,132
Ubisoft,127


Databricks visualization. Run in Databricks to view.

🔧 Top 10 Développeurs :


developer,count
Choice of Games,140
,127
Creobit,122
Laush Dmitriy Sergeevich,108
Sokpop Collective,98
"KOEI TECMO GAMES CO., LTD.",90
Reforged Group,89
Boogygames Studios,80
Hosted Games,79
Elephant Games,75


Databricks visualization. Run in Databricks to view.

In [0]:
# --- 4. Focus COVID : Impact sur les sorties (2018-2022) ---
# Comparaison avant / pendant / après COVID
df_covid = df_clean.filter(col("release_year").between(2018, 2022)) \
                   .groupBy("release_year") \
                   .count() \
                   .orderBy("release_year")

print("🦠 Focus COVID — Sorties 2018-2022 :")
display(df_covid)
# 👉 Action : Bar Chart. X='release_year', Y='count'. Observer le pic 2020-2021.

# Calcul de la variation par rapport à 2019 (dernière année pré-COVID)
from pyspark.sql import Row
covid_rows = {row["release_year"]: row["count"] for row in df_covid.collect()}
baseline = covid_rows.get(2019, 1)
for y in [2020, 2021, 2022]:
    if y in covid_rows:
        variation = ((covid_rows[y] - baseline) / baseline) * 100
        print(f"  {y} vs 2019 : {'+' if variation > 0 else ''}{variation:.1f}%")

🦠 Focus COVID — Sorties 2018-2022 :


release_year,count
2018,5416
2019,5053
2020,6059
2021,6266
2022,5301


Databricks visualization. Run in Databricks to view.

  2020 vs 2019 : +19.9%
  2021 vs 2019 : +24.0%
  2022 vs 2019 : +4.9%


## 💰 Macro-Analyse : Économie et Accessibilité

**Questions :**
* Quelle est la distribution des prix ?
* Quelles sont les langues indispensables pour une sortie mondiale ?
* Quelle est la part de jeux réservés aux adultes (+16/18) ?

In [0]:
# --- 3. Distribution des Prix ---
# Nous filtrons les jeux gratuits (0€) et les valeurs extrêmes (>100€) 
# pour obtenir une distribution lisible du "cœur de marché" (Mainstream Market)
df_prices = df_clean.filter((col("price_eur") > 0) & (col("price_eur") < 100)).select("price_eur")

print("💸 Distribution des prix (Focus marché 1€ - 100€) :")
display(df_prices) 
# 👉 Action : Histogramme. X='price_eur', Bins=20. Tu verras une belle courbe.

# --- 4. Analyse des Langues ---
# Identification des langues critiques pour la localisation (L10n)
df_top_lang = df_languages.groupBy("language") \
                          .count() \
                          .orderBy(desc("count")) \
                          .limit(10)
print("🗣️ Top 10 Langues supportées :")
display(df_top_lang) 
# 👉 Action : Bar Chart horizontal.

# --- 5. Restrictions d'Âge (PEGI/ESRB) ---
# Segmentation de l'audience. Grâce au nettoyage regex, les données sont propres.
df_age = df_clean.withColumn(
    "age_category", 
    when(col("required_age") >= 18, "18+ (Adults Only)")
    .when(col("required_age") >= 16, "16+ (Mature)")
    .otherwise("General Audience (<16)")
).groupBy("age_category").count()

print("🔞 Répartition par catégorie d'âge :")
display(df_age) 
# 👉 Action : Pie Chart.

💸 Distribution des prix (Focus marché 1€ - 100€) :


price_eur
9.99
9.99
19.99
19.99
1.99
19.99
12.99
2.99
13.99
0.99


Databricks visualization. Run in Databricks to view.

🗣️ Top 10 Langues supportées :


language,count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6599


Databricks visualization. Run in Databricks to view.

🔞 Répartition par catégorie d'âge :


age_category,count
18+ (Adults Only),230
16+ (Mature),76
General Audience (<16),55385


Databricks visualization. Run in Databricks to view.

## 🎯 Analyse des Genres : Rentabilité et Qualité

**Questions :**
* Quels sont les genres les plus lucratifs (Prix moyen) ?
* Quels genres obtiennent les meilleures recommandations des joueurs ?
* Les éditeurs ont-ils des genres de prédilection ?

In [0]:
# Agrégation complexe par genre : Volume, Prix Moyen, et Satisfaction Client
# Nous remplaçons 'avg_recommendations' par 'avg_rating' calculé précédemment
df_genre_metrics = df_genres_exploded.groupBy("genre") \
    .agg(
        count("title").alias("volume"),
        round(avg("price_eur"), 2).alias("avg_price"),
        round(avg("rating_pct"), 2).alias("avg_rating"), # Score moyen sur 100
        round(avg("total_reviews"), 0).alias("avg_engagement")
    ) \
    .filter(col("volume") > 100) # Filtre pour la pertinence statistique

# --- 1. Genres les plus représentés (Volume) ---
print("📊 Genres les plus populaires (Volume) :")
display(df_genre_metrics.orderBy(desc("volume")).limit(10))

# --- 2. Genres les plus lucratifs (Prix Moyen) ---
print("💎 Genres les plus chers (Premium) :")
display(df_genre_metrics.orderBy(desc("avg_price")).limit(10))

# --- 3. Genres les mieux notés (Qualité Perçue) ---
print("⭐ Genres avec la meilleure satisfaction moyenne :")
display(df_genre_metrics.orderBy(desc("avg_rating")).limit(10))

📊 Genres les plus populaires (Volume) :


genre,volume,avg_price,avg_rating,avg_engagement
Indie,39681,6.8,73.98,927.0
Action,23759,7.98,72.81,2717.0
Casual,22086,5.8,74.01,524.0
Adventure,21431,8.31,73.76,1649.0
Strategy,10895,8.66,71.81,1450.0
Simulation,10836,9.38,69.19,1659.0
RPG,9534,9.36,72.95,2381.0
Early Access,6145,8.87,70.46,858.0
Free to Play,3393,0.3,72.08,6780.0
Sports,2666,9.26,69.67,1378.0


Databricks visualization. Run in Databricks to view.

💎 Genres les plus chers (Premium) :


genre,volume,avg_price,avg_rating,avg_engagement
Game Development,159,21.36,73.78,193.0
Audio Production,195,20.78,67.18,403.0
Photo Editing,105,20.42,69.66,5633.0
Video Production,247,19.81,67.74,518.0
Software Training,164,19.75,69.6,171.0
Design & Illustration,406,19.54,71.9,1727.0
Animation & Modeling,322,19.11,70.49,2227.0
Education,317,14.59,68.29,85.0
Utilities,682,11.71,69.77,1148.0
Simulation,10836,9.38,69.19,1659.0


Databricks visualization. Run in Databricks to view.

⭐ Genres avec la meilleure satisfaction moyenne :


genre,volume,avg_price,avg_rating,avg_engagement
Casual,22086,5.8,74.01,524.0
Indie,39681,6.8,73.98,927.0
Game Development,159,21.36,73.78,193.0
Adventure,21431,8.31,73.76,1649.0
RPG,9534,9.36,72.95,2381.0
Action,23759,7.98,72.81,2717.0
Free to Play,3393,0.3,72.08,6780.0
Design & Illustration,406,19.54,71.9,1727.0
Strategy,10895,8.66,71.81,1450.0
Animation & Modeling,322,19.11,70.49,2227.0


Databricks visualization. Run in Databricks to view.

## 🖥️ Analyse Technique : Plateformes (OS)

**Question :** Y a-t-il des genres qui privilégient certaines plateformes (Linux/Mac) ?
Cette analyse croisée aide à décider du portage technique du jeu.

In [0]:
# --- Distribution globale par plateforme ---
# Combien de jeux sont disponibles sur chaque OS ?
import builtins  # Pour utiliser le round() Python natif (celui de PySpark écrase le builtin)

total_games = df_clean.count()

df_os_counts = df_clean.select(
    _sum(when(col("platforms.windows") == True, 1).otherwise(0)).alias("Windows"),
    _sum(when(col("platforms.mac") == True, 1).otherwise(0)).alias("Mac"),
    _sum(when(col("platforms.linux") == True, 1).otherwise(0)).alias("Linux"),
    lit(total_games).alias("Total")
)

print("🖥️ Disponibilité par plateforme :")
display(df_os_counts)

# Version longue pour graphique
from pyspark.sql import Row
os_row = df_os_counts.first()
os_data = [
    Row(plateforme="Windows", nb_jeux=os_row["Windows"], pct=builtins.round(os_row["Windows"]/total_games*100, 1)),
    Row(plateforme="Mac", nb_jeux=os_row["Mac"], pct=builtins.round(os_row["Mac"]/total_games*100, 1)),
    Row(plateforme="Linux", nb_jeux=os_row["Linux"], pct=builtins.round(os_row["Linux"]/total_games*100, 1))
]
df_os_pivot = spark.createDataFrame(os_data)

print("📊 Parts de marché par OS :")
display(df_os_pivot)
# 👉 Action : Pie Chart ou Bar Chart. X='plateforme', Y='nb_jeux'

🖥️ Disponibilité par plateforme :


Windows,Mac,Linux,Total
55676,12770,8458,55691


📊 Parts de marché par OS :


plateforme,nb_jeux,pct
Windows,55676,100.0
Mac,12770,22.9
Linux,8458,15.2


Databricks visualization. Run in Databricks to view.

In [0]:
# --- Analyse croisée : Support OS par Genre ---
# Question : Do certain genres tend to be preferentially available on certain platforms?
df_platform_genre = df_genres_exploded.select(
    col("genre"),
    when(col("platforms.linux") == True, 1).otherwise(0).alias("is_linux"),
    when(col("platforms.mac") == True, 1).otherwise(0).alias("is_mac")
).groupBy("genre").agg(
    round(avg("is_linux") * 100, 2).alias("linux_support_pct"),
    round(avg("is_mac") * 100, 2).alias("mac_support_pct"),
    count("*").alias("total_games")
).filter(col("total_games") > 500)

print("🐧 Support Linux & Mac par Genre :")
display(df_platform_genre.orderBy(desc("linux_support_pct")))
# 👉 Action : Grouped Bar Chart. X='genre', Y='linux_support_pct' + 'mac_support_pct'

🐧 Support Linux & Mac par Genre :


genre,linux_support_pct,mac_support_pct,total_games
Indie,17.59,25.04,39681
Strategy,16.76,27.58,10895
RPG,15.98,23.58,9534
Adventure,15.41,23.51,21431
Casual,14.96,23.23,22086
Action,14.22,19.21,23759
Simulation,14.14,22.51,10836
Racing,14.11,19.68,2155
Free to Play,13.97,24.9,3393
Massively Multiplayer,11.23,18.49,1460


Databricks visualization. Run in Databricks to view.

## 🌟 Analyses Complémentaires

Pour répondre exhaustivement à toutes les demandes de l'énoncé :
1.  **Hall of Fame :** Les 10 jeux les mieux notés (avec un minimum de votes significatif).
2.  **Stratégie Promotionnelle :** Analyse du volume de jeux avec remise (Discount).
3.  **ADN des Éditeurs :** Spécialisation des Top Publishers par genre (question de l'énoncé).
4.  **ADN des Studios :** Spécialisation des Top Développeurs par genre.

In [0]:
# --- 1. Top 10 des Meilleurs Jeux (Hall of Fame) ---
# Filtre > 5000 avis pour éviter les jeux inconnus avec 1 seul avis positif (100%)
import builtins

df_hall_of_fame = df_clean.filter(col("total_reviews") > 5000) \
                          .select("title", "rating_pct", "total_reviews", "release_year") \
                          .orderBy(desc("rating_pct")) \
                          .limit(10)

print("🏆 Hall of Fame — Les 10 jeux les mieux notés :")
display(df_hall_of_fame)

# --- 2. Analyse des Promotions (Discount) ---
df_discount_raw = df_raw.select(col("data.discount").alias("discount_raw"))

# Comptage : jeux avec une remise active (discount != "0" et non null)
total_with_data = df_discount_raw.filter(col("discount_raw").isNotNull()).count()
discounted = df_discount_raw.filter(
    (col("discount_raw").isNotNull()) & 
    (col("discount_raw") != "0") & 
    (col("discount_raw") != 0)
).count()

pct_discount = builtins.round(discounted / max(total_with_data, 1) * 100, 1)
print(f"🏷️ Promotions : {discounted} jeux en promo sur {total_with_data} ({pct_discount}%)")
print(f"   → {total_with_data - discounted} jeux au prix normal")

# --- 3. ADN des Éditeurs / Publishers (question de l'énoncé) ---
# "Do some publishers have favorite genres?"
top_pub_names = [row['publisher'] for row in df_top_publishers.limit(5).collect()]

df_pub_dna = df_clean.filter(col("publisher").isin(top_pub_names)) \
                     .select("publisher", explode(split(col("genre_string"), ", ")).alias("genre")) \
                     .groupBy("publisher", "genre") \
                     .count() \
                     .orderBy("publisher", desc("count"))

print("🧬 ADN des Éditeurs : Genres favoris des Top Publishers :")
display(df_pub_dna)
# 👉 Action : Bar Chart Stacked. X='publisher', Y='count', Group=genre

# --- 4. ADN des Studios / Développeurs ---
top_dev_names = [row['developer'] for row in df_top_devs.limit(5).collect()]

df_dev_dna = df_clean.filter(col("developer").isin(top_dev_names)) \
                     .select("developer", explode(split(col("genre_string"), ", ")).alias("genre")) \
                     .groupBy("developer", "genre") \
                     .count() \
                     .orderBy("developer", desc("count"))

print("🔧 ADN des Studios : Genres favoris des Top Développeurs :")
display(df_dev_dna)

🏆 Hall of Fame — Les 10 jeux les mieux notés :


title,rating_pct,total_reviews,release_year
Aseprite,99.33,11903,2016
A Short Hike,99.26,11732,2019
Senren＊Banka,99.21,10677,2020
Our Life: Beginnings & Always,98.88,8043,2020
People Playground,98.86,144569,2019
Portal 2,98.78,309441,2011
Vampire Survivors,98.77,131935,2022
The Room 4: Old Sins,98.75,10424,2021
ATRI -My Dear Moments-,98.74,9124,2020
The Henry Stickmin Collection,98.72,33340,null


🏷️ Promotions : 2518 jeux en promo sur 55691 (4.5%)
   → 53173 jeux au prix normal
🧬 ADN des Éditeurs : Genres favoris des Top Publishers :


publisher,genre,count
8floor,Casual,202
8floor,Strategy,22
8floor,Simulation,10
8floor,Adventure,8
8floor,Indie,1
Big Fish Games,Casual,418
Big Fish Games,Adventure,392
Big Fish Games,Indie,7
Big Fish Games,Simulation,7
Big Fish Games,Strategy,6


🔧 ADN des Studios : Genres favoris des Top Développeurs :


developer,genre,count
,,83
,Indie,24
,Action,22
,Casual,16
,Adventure,14
,RPG,9
,Simulation,9
,Strategy,7
,Early Access,2
,Sports,2


Databricks visualization. Run in Databricks to view.